# 01 — Analyse exploratoire : paludisme en Guinée

Ce notebook charge `data/raw/paludisme.csv`, décrit les variables, vérifie la couverture géographique/temporelle et explore les distributions météo par niveau de risque (`faible`, `moyen`, `eleve`).


# Prédiction du risque de paludisme en Guinée — CRISP-DM

Notebook adapté au fichier `paludisme_1786569715919.csv`.

**Objectif métier :** estimer un niveau de risque (`faible`, `moyen`, `eleve`) à partir des conditions météorologiques et environnementales afin d'aider un centre de santé à anticiper les périodes de forte demande.

> **Important :** le CSV fourni contient des données synthétiques. Les colonnes `cas_paludisme_simules` et `taux_incidence_simule_pour_1000` sont retirées des variables explicatives pour éviter une fuite de données, car elles sont directement liées au risque cible.
    

## Installation éventuelle

Le notebook utilise `pandas`, `numpy`, `matplotlib`, `seaborn`, `scikit-learn` et `xgboost`.

Si une bibliothèque manque dans votre environnement Jupyter, exécutez dans une cellule :

```python
%pip install pandas numpy matplotlib seaborn scikit-learn xgboost
```
    

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")
RANDOM_STATE = 42
pd.set_option("display.max_columns", 60)
print("Bibliothèques chargées.")
    

In [ ]:
from pathlib import Path

# Chemin relatif au dossier notebooks/ dans le depot GitHub
file_candidates = [
    Path("../data/raw/paludisme.csv"),
    Path("data/raw/paludisme.csv"),
    Path("paludisme.csv"),
]
DATA_PATH = next((path for path in file_candidates if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Fichier introuvable. Lancez ce notebook depuis notebooks/ dans le depot cloné.")

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
print(f"Fichier charge : {DATA_PATH}")
print(f"Dimensions : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
df.head()


## 1. Compréhension des données

In [ ]:
print("Types et valeurs non nulles :")
df.info()

print("\nRésumé statistique des variables numériques :")
display(df.describe(include="all").T)

print("\nValeurs manquantes :")
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("valeurs_manquantes"))

print("\nDoublons complets :", int(df.duplicated().sum()))
print("\nRépartition de la variable cible :")
display(df["niveau_risque"].value_counts().rename_axis("niveau_risque").to_frame("effectif"))
    

In [ ]:
# Contrôles de couverture géographique et temporelle.
print("Régions administratives :", df["region_administrative"].nunique())
display(df["region_administrative"].value_counts().rename_axis("region").to_frame("observations"))

print("Nombre de préfectures :", df["prefecture"].nunique())
print("Période couverte :", df["date"].min(), "à", df["date"].max())
    

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df, x="niveau_risque", order=["faible", "moyen", "eleve"], ax=axes[0])
axes[0].set_title("Répartition du niveau de risque")
axes[0].set_xlabel("Niveau de risque")
axes[0].set_ylabel("Nombre d'observations")

region_risk = pd.crosstab(df["region_administrative"], df["niveau_risque"], normalize="index")
region_risk = region_risk.reindex(columns=["faible", "moyen", "eleve"], fill_value=0)
region_risk.plot(kind="bar", stacked=True, ax=axes[1], color=["#5abf90", "#f2c14e", "#e76f51"])
axes[1].set_title("Risque relatif par région")
axes[1].set_xlabel("")
axes[1].set_ylabel("Proportion")
axes[1].legend(title="Risque", loc="upper right")
plt.tight_layout()
plt.show()
    

In [ ]:
# Distributions des principales variables météo.
weather_cols = [
    "temperature_moyenne_c", "precipitation_7j_mm", "precipitation_30j_mm",
    "humidite_relative_pct", "humidite_sol_pct", "indice_eau_stagnante",
]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, column in zip(axes.ravel(), weather_cols):
    sns.histplot(data=df, x=column, hue="niveau_risque", bins=25, element="step", stat="density", common_norm=False, ax=ax)
    ax.set_title(column)
plt.tight_layout()
plt.show()
    

In [ ]:
# Corrélation entre les variables numériques.
numeric_for_corr = df.select_dtypes(include=np.number).columns
corr = df[numeric_for_corr].corr()
plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.3)
plt.title("Matrice de corrélation des variables numériques")
plt.tight_layout()
plt.show()
    